# LangChain — Think and Self-Reflection 示例

**Self-Reflection（自我反思）** 是一种高级推理模式：LLM 先生成初步回答，然后对自己的回答进行批判性审视，最后基于反思结果生成改进后的回答。

## 工作流程

```
问题 → [Step 1: 初步思考] → 初步回答
                                ↓
                        [Step 2: 自我反思] → 发现问题/不足
                                ↓
                        [Step 3: 改进回答] → 最终答案
```

**优势：**
- 减少幻觉和事实错误
- 提高回答的完整性和准确性
- 模拟人类"先想、再检查、再修正"的思维过程

**模型：** 通义千问 (Qwen) via DashScope API

In [ ]:
import sys
!{sys.executable} -m pip install -q langchain langchain-core langchain-openai python-dotenv

## 1. 配置模型

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("../.env")

llm = ChatOpenAI(
    model="qwen-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.7,
)

response = llm.invoke("你好，请用一句话介绍你自己。")
print(response.content)

## 2. 定义三阶段 Prompt

Self-Reflection 的核心是三个 Prompt，分别对应：
1. **初步思考**（Think）：针对问题生成详细的初步回答
2. **自我反思**（Reflect）：批判性审视初步回答，找出不足
3. **改进回答**（Refine）：根据反思结果生成最终改进版答案

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1: 初步思考
think_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个善于深度思考的AI助手。请仔细分析问题，给出详细、有条理的回答。"),
    ("human", "问题：{question}\n\n请深入思考并给出你的回答："),
])

# Step 2: 自我反思
reflect_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是一个严格的审稿人。你的任务是批判性地审视一个AI的回答，"
     "找出其中的错误、遗漏、逻辑漏洞、或可以改进的地方。\n"
     "请从以下维度进行反思：\n"
     "1. **事实准确性**：有没有事实错误或不确定的陈述？\n"
     "2. **完整性**：有没有遗漏重要的方面？\n"
     "3. **逻辑性**：推理过程是否严密？\n"
     "4. **表达清晰度**：回答是否清晰易懂？"),
    ("human",
     "原始问题：{question}\n\n"
     "AI的初步回答：\n{initial_answer}\n\n"
     "请对以上回答进行严格的自我反思和批判性分析："),
])

# Step 3: 改进回答
refine_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是一个追求卓越的AI助手。根据之前的反思意见，生成一个改进后的最终回答。"
     "改进后的回答应该修正错误、补充遗漏、加强逻辑、提高清晰度。"),
    ("human",
     "原始问题：{question}\n\n"
     "初步回答：\n{initial_answer}\n\n"
     "反思意见：\n{reflection}\n\n"
     "请基于以上反思，生成改进后的最终回答："),
])

print("三阶段 Prompt 定义完成")

## 3. 构建 Self-Reflection Chain

使用 LCEL（LangChain Expression Language）将三个阶段串联成一个完整的 Chain。

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

think_chain = think_prompt | llm | parser
reflect_chain = reflect_prompt | llm | parser
refine_chain = refine_prompt | llm | parser


def self_reflect(question: str, verbose: bool = True) -> dict:
    """Execute the full Think → Reflect → Refine pipeline."""

    # Step 1: 初步思考
    if verbose:
        print("=" * 60)
        print("📝 Step 1: 初步思考")
        print("=" * 60)
    initial_answer = think_chain.invoke({"question": question})
    if verbose:
        print(initial_answer)

    # Step 2: 自我反思
    if verbose:
        print("\n" + "=" * 60)
        print("🔍 Step 2: 自我反思")
        print("=" * 60)
    reflection = reflect_chain.invoke({
        "question": question,
        "initial_answer": initial_answer,
    })
    if verbose:
        print(reflection)

    # Step 3: 改进回答
    if verbose:
        print("\n" + "=" * 60)
        print("✅ Step 3: 改进后的最终回答")
        print("=" * 60)
    final_answer = refine_chain.invoke({
        "question": question,
        "initial_answer": initial_answer,
        "reflection": reflection,
    })
    if verbose:
        print(final_answer)

    return {
        "question": question,
        "initial_answer": initial_answer,
        "reflection": reflection,
        "final_answer": final_answer,
    }

print("self_reflect 函数定义完成")

## 4. 用 LCEL Chain 实现（纯链式写法）

除了上面的函数写法，也可以用 LCEL 的 `RunnablePassthrough.assign` 把三步串成一条链。

In [ ]:
reflection_chain = (
    # 输入: {"question": "..."}
    RunnablePassthrough.assign(
        initial_answer=think_chain
    )
    # 此时: {"question": "...", "initial_answer": "..."}
    | RunnablePassthrough.assign(
        reflection=reflect_chain
    )
    # 此时: {"question": "...", "initial_answer": "...", "reflection": "..."}
    | RunnablePassthrough.assign(
        final_answer=refine_chain
    )
    # 最终: {"question": "...", "initial_answer": "...", "reflection": "...", "final_answer": "..."}
)

print("LCEL reflection_chain 构建完成")
print("输入格式: {'question': '你的问题'}")
print("输出格式: {'question', 'initial_answer', 'reflection', 'final_answer'}")

## 5. 示例：函数式调用（带详细输出）

In [ ]:
result = self_reflect("为什么天空是蓝色的？请从物理学角度解释。")

## 6. 示例：LCEL Chain 调用

In [ ]:
result = reflection_chain.invoke({"question": "Python 的 GIL 是什么？它对多线程编程有什么影响？"})

print("=" * 60)
print("📝 初步回答：")
print("=" * 60)
print(result["initial_answer"])
print("\n" + "=" * 60)
print("🔍 反思意见：")
print("=" * 60)
print(result["reflection"])
print("\n" + "=" * 60)
print("✅ 改进后的最终回答：")
print("=" * 60)
print(result["final_answer"])

## 7. 多轮反思（Iterative Reflection）

有时一轮反思不够，可以迭代多次，每轮都在上一轮的基础上继续改进。

In [ ]:
def iterative_reflect(question: str, rounds: int = 2) -> dict:
    """Multi-round self-reflection: each round refines the previous answer."""

    print("=" * 60)
    print(f"📝 初步思考")
    print("=" * 60)
    current_answer = think_chain.invoke({"question": question})
    print(current_answer)

    for i in range(rounds):
        print(f"\n{'=' * 60}")
        print(f"🔍 第 {i + 1} 轮反思")
        print("=" * 60)
        reflection = reflect_chain.invoke({
            "question": question,
            "initial_answer": current_answer,
        })
        print(reflection)

        print(f"\n{'=' * 60}")
        print(f"✅ 第 {i + 1} 轮改进")
        print("=" * 60)
        current_answer = refine_chain.invoke({
            "question": question,
            "initial_answer": current_answer,
            "reflection": reflection,
        })
        print(current_answer)

    return {"question": question, "final_answer": current_answer}

print("iterative_reflect 函数定义完成")

In [ ]:
result = iterative_reflect(
    "设计一个高并发的短链接服务系统，需要考虑哪些关键技术点？",
    rounds=2,
)

## 8. 对比：直接回答 vs 经过反思

对同一个问题，比较"一次性回答"和"经过反思后的回答"的质量差异。

In [ ]:
question = "比较 REST API 和 GraphQL 的优缺点，什么场景该用哪个？"

print("=" * 60)
print("【方式A】直接回答（无反思）")
print("=" * 60)
direct_answer = think_chain.invoke({"question": question})
print(direct_answer)

print("\n\n" + "#" * 60)
print("【方式B】经过反思后的回答")
print("#" * 60)
result = self_reflect(question, verbose=False)
print(result["final_answer"])